In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

from matplotlib import colors, cm
from nilearn import plotting


# ============================================================
# USER PATHS
# ============================================================

# Folder containing:
#   atom_00_*.func.gii
#   atom_01_*.func.gii
#   ...
#   k_hubness_*.func.gii
KMAP_DIR = Path(
    "/lustre06/project/6002437/mahdi199/thomas_comparision/output/"
    "HC043_ses-01_hemi-LR_surf-fsLR-32k/KMAP_HC043_ses-01_hemi-LR_surf-fsLR-32k"
)

# Surface geometry files.
# Inflated surfaces are usually easiest to inspect visually.
LH_SURF = Path("/lustre06/project/6002437/mahdi199/thomas_comparision/fsLR-32k.L.inflated.surf.gii")
RH_SURF = Path("/lustre06/project/6002437/mahdi199/thomas_comparision/fsLR-32k.R.inflated.surf.gii")

# Optional sulcal maps.
# Leave as None if you do not have them.
LH_SULC = None
RH_SULC = None


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

# Saved inside the same folder where the atoms exist
FIG_DIR = KMAP_DIR / "surface_plots"
FIG_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Figures will be saved to:")
print(FIG_DIR)


# ============================================================
# HELPERS
# ============================================================

def load_gifti_vector(path):
    """
    Load a single-map GIFTI as a 1D NumPy vector.
    """

    img = nib.load(
        str(path)
    )

    if len(img.darrays) == 0:
        raise ValueError(
            f"No DataArrays found in: {path}"
        )

    if len(img.darrays) == 1:

        data = np.asarray(
            img.darrays[0].data
        )

    else:

        data = np.asarray(
            img.agg_data()
        )

    return np.asarray(
        data
    ).squeeze()


def get_surface_nvertices(surface_path):
    """
    Return number of vertices in a .surf.gii mesh.
    """

    img = nib.load(
        str(surface_path)
    )

    coords = img.agg_data(
        "pointset"
    )

    if isinstance(
        coords,
        tuple
    ):

        if len(coords) == 0:

            raise ValueError(
                f"No POINTSET coordinates found in {surface_path}"
            )

        coords = coords[0]

    coords = np.asarray(
        coords
    )

    return coords.shape[0]


def load_background(path):
    """
    Load optional sulcal/background map.
    """

    if path is None:
        return None

    return load_gifti_vector(
        path
    )


# ============================================================
# SURFACE DIMENSIONS
# ============================================================

n_lh = get_surface_nvertices(
    LH_SURF
)

n_rh = get_surface_nvertices(
    RH_SURF
)

print("\nSurface geometry:")
print(
    f"Left hemisphere : {n_lh:,} vertices"
)
print(
    f"Right hemisphere: {n_rh:,} vertices"
)
print(
    f"Total           : {n_lh + n_rh:,} vertices"
)


# ============================================================
# OPTIONAL BACKGROUND MAPS
# ============================================================

lh_bg = load_background(
    LH_SULC
)

rh_bg = load_background(
    RH_SULC
)

if (
    lh_bg is not None
    and len(lh_bg) != n_lh
):

    raise ValueError(
        f"LH background has {len(lh_bg)} values "
        f"but LH surface has {n_lh} vertices."
    )

if (
    rh_bg is not None
    and len(rh_bg) != n_rh
):

    raise ValueError(
        f"RH background has {len(rh_bg)} values "
        f"but RH surface has {n_rh} vertices."
    )


# ============================================================
# MAIN PLOTTING FUNCTION
# ============================================================

def plot_spark_surface(
    gifti_path,
    output_dir,
    is_hubness=False,
    dpi=200
):
    """
    Plot one bilateral SPARK GIFTI map as:

        LH lateral | LH medial | RH lateral | RH medial

    Saves PNG only.
    """

    gifti_path = Path(
        gifti_path
    )

    values = load_gifti_vector(
        gifti_path
    )

    expected = (
        n_lh + n_rh
    )

    if values.size != expected:

        raise ValueError(
            f"\nVertex count mismatch for:\n"
            f"{gifti_path}\n"
            f"GIFTI values : {values.size:,}\n"
            f"LH + RH mesh : {expected:,}\n"
            f"({n_lh:,} + {n_rh:,})"
        )


    # ========================================================
    # SPLIT LH / RH
    #
    # Confirmed ordering:
    # [all LH vertices] + [all RH vertices]
    # ========================================================

    lh_data = values[
        :n_lh
    ]

    rh_data = values[
        n_lh:
    ]


    # ========================================================
    # FINITE VALUES
    # ========================================================

    finite_values = values[
        np.isfinite(values)
    ]

    if finite_values.size == 0:

        raise ValueError(
            f"No finite values found in {gifti_path}"
        )


    # ========================================================
    # COLOR SCALE
    # ========================================================

    if is_hubness:

        vmin = 0.0

        vmax = float(
            np.max(
                finite_values
            )
        )

        if vmax == 0:
            vmax = 1.0

        cmap = "viridis"

        norm = colors.Normalize(
            vmin=vmin,
            vmax=vmax
        )

        symmetric_cbar = False

    else:

        vmax = float(
            np.max(
                np.abs(
                    finite_values
                )
            )
        )

        if vmax == 0:
            vmax = 1.0

        vmin = -vmax

        cmap = "RdBu_r"

        norm = colors.Normalize(
            vmin=vmin,
            vmax=vmax
        )

        symmetric_cbar = True


    # ========================================================
    # COMPACT FIGURE
    # ========================================================

    fig = plt.figure(
        figsize=(12, 3.2)
    )

    gs = fig.add_gridspec(
        1,
        5,
        width_ratios=[
            1,
            1,
            1,
            1,
            0.045
        ],
        wspace=0.01
    )


    # ========================================================
    # CREATE 4 SURFACE AXES
    # ========================================================

    ax1 = fig.add_subplot(
        gs[0, 0],
        projection="3d"
    )

    ax2 = fig.add_subplot(
        gs[0, 1],
        projection="3d"
    )

    ax3 = fig.add_subplot(
        gs[0, 2],
        projection="3d"
    )

    ax4 = fig.add_subplot(
        gs[0, 3],
        projection="3d"
    )

    axes = [
        ax1,
        ax2,
        ax3,
        ax4
    ]


    # ========================================================
    # VIEW DEFINITIONS
    # ========================================================

    views = [

        (
            "left",
            "lateral",
            lh_data,
            LH_SURF,
            lh_bg,
            "LH lateral"
        ),

        (
            "left",
            "medial",
            lh_data,
            LH_SURF,
            lh_bg,
            "LH medial"
        ),

        (
            "right",
            "lateral",
            rh_data,
            RH_SURF,
            rh_bg,
            "RH lateral"
        ),

        (
            "right",
            "medial",
            rh_data,
            RH_SURF,
            rh_bg,
            "RH medial"
        ),
    ]


    # ========================================================
    # PLOT SURFACES
    # ========================================================

    for ax, (
        hemi,
        view,
        hemi_data,
        surface,
        background,
        title
    ) in zip(
        axes,
        views
    ):

        plotting.plot_surf_stat_map(

            surf_mesh=str(
                surface
            ),

            stat_map=hemi_data,

            bg_map=background,

            hemi=hemi,

            view=view,

            cmap=cmap,

            # We create our own shared colorbar
            colorbar=False,

            symmetric_cbar=symmetric_cbar,

            vmin=vmin,

            vmax=vmax,

            threshold=None,

            bg_on_data=(
                background
                is not None
            ),

            axes=ax,

            figure=fig,

            engine="matplotlib",
        )

        ax.set_title(
            title,
            fontsize=10,
            pad=-2
        )


    # ========================================================
    # SHARED COLORBAR
    # ========================================================

    cax = fig.add_subplot(
        gs[0, 4]
    )

    scalar_map = cm.ScalarMappable(
        norm=norm,
        cmap=cmap
    )

    scalar_map.set_array([])

    cbar = fig.colorbar(
        scalar_map,
        cax=cax
    )

    cbar.ax.tick_params(
        labelsize=8
    )

    if is_hubness:

        cbar.set_label(
            "k-hubness",
            fontsize=9
        )

    else:

        cbar.set_label(
            "Z score",
            fontsize=9
        )


    # ========================================================
    # TITLE
    # ========================================================

    clean_title = (
        gifti_path.name
    )

    if clean_title.endswith(
        ".func.gii"
    ):

        clean_title = clean_title[
            :-9
        ]

    fig.suptitle(
        clean_title,
        fontsize=12,
        y=0.98
    )


    # ========================================================
    # REMOVE EXCESS WHITESPACE
    # ========================================================

    fig.subplots_adjust(
        left=0.01,
        right=0.97,
        bottom=0.01,
        top=0.88
    )


    # ========================================================
    # SAVE PNG ONLY
    # ========================================================

    png_path = (
        output_dir
        / f"{clean_title}.png"
    )

    fig.savefig(
        png_path,
        dpi=dpi,
        bbox_inches="tight",
        pad_inches=0.03
    )

    plt.show()

    plt.close(
        fig
    )

    print(
        f"Saved: {png_path.name}"
    )

    return {

        "file":
            gifti_path.name,

        "min":
            float(
                np.min(
                    finite_values
                )
            ),

        "max":
            float(
                np.max(
                    finite_values
                )
            ),

        "nonzero":
            int(
                np.count_nonzero(
                    values
                )
            ),

        "png":
            str(
                png_path
            ),
    }


# ============================================================
# FIND SPARK OUTPUT FILES
# ============================================================

atom_files = sorted(
    KMAP_DIR.glob(
        "atom_*.func.gii"
    )
)

hub_files = sorted(
    KMAP_DIR.glob(
        "k_hubness_*.func.gii"
    )
)

print("\nFiles found:")
print(
    f"Atoms     : {len(atom_files)}"
)
print(
    f"K-hubness : {len(hub_files)}"
)


# ============================================================
# PLOT ALL ATOMS
# ============================================================

results = []

for atom_file in atom_files:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "Plotting:",
        atom_file.name
    )

    result = plot_spark_surface(
        gifti_path=atom_file,
        output_dir=FIG_DIR,
        is_hubness=False
    )

    results.append(
        result
    )


# ============================================================
# PLOT K-HUBNESS
# ============================================================

for hub_file in hub_files:

    print(
        "\n"
        + "=" * 70
    )

    print(
        "Plotting:",
        hub_file.name
    )

    result = plot_spark_surface(
        gifti_path=hub_file,
        output_dir=FIG_DIR,
        is_hubness=True
    )

    results.append(
        result
    )


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "DONE"
)

print(
    "=" * 70
)

print(
    f"Generated plots for {len(results)} SPARK map(s)."
)

print(
    f"Saved in:\n{FIG_DIR}"
)

print(
    "\nSummary:"
)

for r in results:

    print(
        f"{r['file']:<55} "
        f"nonzero={r['nonzero']:>6,}  "
        f"range=[{r['min']:.3f}, {r['max']:.3f}]"
    )